# Alpamayo 1.5: Local Offline Inference Demo

This notebook runs Alpamayo 1.5 on a local offline clip directory and performs offline inference.

This FINAL version assumes the loader already returns xyz and rotations in a fully unified model/action frame.
Therefore, plotting and metric computation should use the returned xyz directly without extra eval-side xy rotation.

In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print('cwd =', Path.cwd().resolve())
print('repo_root =', repo_root)
print('src_path =', src_path)
print('src exists =', src_path.exists())
print('PYTHONPATH =', os.environ.get('PYTHONPATH'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from transformers import BitsAndBytesConfig

from alpamayo1_5 import helper
from alpamayo1_5.load_offline_dataset import load_offline_dataset
from alpamayo1_5.models.alpamayo1_5 import Alpamayo1_5

In [ ]:
# ===== Local paths =====
CLIP_DIR = Path('/workspace/dataset/')
MODEL_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Alpamayo-1.5-10B/snapshots/f11cd25b758ab560114019b555dde2a8b92d88b4')
PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--Qwen--Qwen3-VL-8B-Instruct/snapshots/0c351dd01ed87e9c1b53cbc748cba10e6187ff3b')
COSMOS_PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Cosmos-Reason2-8B/snapshots/f715d875a8a99a0a2b65aa28633afd9417e46bd9')

# ===== Inference config =====
DEVICE = 'cuda'
T0_SOD = 175500.23
NUM_HISTORY_STEPS = 16
NUM_FUTURE_STEPS = 64
TIME_STEP = 0.1
NUM_FRAMES = 4
FPS = 30.0
FRAME0_GPS_TIME_SOD = 175484.98
NUM_TRAJ_SAMPLES = 1
TEMPERATURE = 0.6
TOP_P = 0.98
MAX_GENERATION_LENGTH = 256
IMAGE_SIZE = (448, 800)

os.environ['ALPAMAYO_VLM_PROCESSOR_PATH'] = str(COSMOS_PROCESSOR_PATH)

print('CLIP_DIR =', CLIP_DIR)
print('MODEL_PATH =', MODEL_PATH)
print('PROCESSOR_PATH =', PROCESSOR_PATH)
print('COSMOS_PROCESSOR_PATH =', COSMOS_PROCESSOR_PATH)
print('ALPAMAYO_VLM_PROCESSOR_PATH =', os.environ['ALPAMAYO_VLM_PROCESSOR_PATH'])
print('T0_SOD =', T0_SOD)
print('IMAGE_SIZE =', IMAGE_SIZE)

### Path validation

In [ ]:
def _describe_path(p: Path, name: str):
    print(f'{name}: {p}')
    print(f'  exists: {p.exists()}')
    print(f'  is_dir: {p.is_dir() if p.exists() else "N/A"}')

_describe_path(CLIP_DIR, 'CLIP_DIR')
_describe_path(MODEL_PATH, 'MODEL_PATH')
_describe_path(PROCESSOR_PATH, 'PROCESSOR_PATH')
_describe_path(COSMOS_PROCESSOR_PATH, 'COSMOS_PROCESSOR_PATH')

if not CLIP_DIR.exists():
    raise FileNotFoundError(f'CLIP_DIR does not exist: {CLIP_DIR}')
if not MODEL_PATH.exists():
    raise FileNotFoundError(f'MODEL_PATH does not exist: {MODEL_PATH}')
if not PROCESSOR_PATH.exists():
    raise FileNotFoundError(f'PROCESSOR_PATH does not exist: {PROCESSOR_PATH}')
if not COSMOS_PROCESSOR_PATH.exists():
    raise FileNotFoundError(f'COSMOS_PROCESSOR_PATH does not exist: {COSMOS_PROCESSOR_PATH}')
if not (MODEL_PATH / 'config.json').exists():
    raise FileNotFoundError(f'Missing config.json under MODEL_PATH: {MODEL_PATH}')

### Coordinate-frame notes

This FINAL notebook version uses a loader that already returns a unified model/action frame.

1. **global ENU**
   - Intermediate frame derived from lat/lon/alt inside `load_offline_dataset.py`
   - x=east, y=north, z=up

2. **t0 local frame**
   - Internal intermediate frame centered at ego pose of `t0`
   - Used only inside the loader

3. **model/action frame**
   - FINAL returned frame from the loader
   - `ego_history_xyz`, `ego_history_rot`, `ego_future_xyz`, `ego_future_rot` are all in this same frame

Important:
- No extra eval-side xy rotation is applied in this notebook.
- Plotting and metric computation use the returned xyz directly.

### Helper functions

In [ ]:
def wrap_to_pi(x):
    return (x + np.pi) % (2 * np.pi) - np.pi


def wrap_to_180_deg(x):
    return (x + 180.0) % 360.0 - 180.0


def global_motion_yaw_deg(xyz):
    xy = xyz[:, :2]
    disp = xy[-1] - xy[0]
    if np.linalg.norm(disp) < 1e-6:
        return np.nan
    return float(np.rad2deg(np.arctan2(disp[1], disp[0])))


def mean_speed_mps(xyz, dt):
    xy = xyz[:, :2]
    dxy = xy[1:] - xy[:-1]
    step_dist = np.linalg.norm(dxy, axis=1)
    if len(step_dist) == 0:
        return 0.0
    return float(step_dist.mean() / dt)


def yaw_from_rot_plus_x_deg(rot_mats):
    return np.rad2deg(np.arctan2(rot_mats[:, 1, 0], rot_mats[:, 0, 0]))


def history_consistency_table(hist_xyz, hist_rot, dt):
    dxy = hist_xyz[1:, :2] - hist_xyz[:-1, :2]
    step_speed = np.linalg.norm(dxy, axis=1) / dt
    dxy_yaw_deg = np.rad2deg(np.arctan2(dxy[:, 1], dxy[:, 0]))
    rot_yaw_deg = yaw_from_rot_plus_x_deg(hist_rot)[1:]
    yaw_minus_dxy_deg = wrap_to_180_deg(rot_yaw_deg - dxy_yaw_deg)

    return pd.DataFrame({
        'step_idx': np.arange(len(dxy)),
        'dx': dxy[:, 0],
        'dy': dxy[:, 1],
        'step_speed_mps': step_speed,
        'dxy_yaw_deg': dxy_yaw_deg,
        'rot_yaw_deg': rot_yaw_deg,
        'yaw_minus_dxy_deg': yaw_minus_dxy_deg,
        'abs_yaw_minus_dxy_deg': np.abs(yaw_minus_dxy_deg),
    })


def segment_mean_speed_table(gt_xyz, model_xyz, dt, segment_sec=1.0):
    seg_len = int(round(segment_sec / dt))
    rows = []
    start = 0
    seg_id = 0

    while start < len(gt_xyz) - 1:
        end = min(start + seg_len + 1, len(gt_xyz))

        def _mean_speed(xyz):
            seg = xyz[start:end]
            if len(seg) < 2:
                return np.nan
            dxy = seg[1:, :2] - seg[:-1, :2]
            return float(np.linalg.norm(dxy, axis=1).mean() / dt)

        rows.append({
            'segment_id': seg_id,
            't_start_sec': round(start * dt, 2),
            't_end_sec': round((end - 1) * dt, 2),
            'gt_mean_speed_mps': _mean_speed(gt_xyz),
            'model_mean_speed_mps': _mean_speed(model_xyz),
        })

        start += seg_len
        seg_id += 1

    return pd.DataFrame(rows)


def summarize_main_metrics(gt_xyz, pred_xyz_np, dt, cot_value):
    gt_xy = gt_xyz[:, :2].T
    pred_xy = pred_xyz_np[:, :, :2].transpose(0, 2, 1)
    diff = np.linalg.norm(pred_xy - gt_xy[None, ...], axis=1).mean(-1)
    best_idx = int(diff.argmin())
    min_ade = float(diff.min())
    pred_best_xyz = pred_xyz_np[best_idx]

    gt_final_xy = gt_xyz[-1, :2]
    pred_final_xy = pred_best_xyz[-1, :2]
    final_point_error = float(np.linalg.norm(pred_final_xy - gt_final_xy))

    gt_mean_speed = mean_speed_mps(gt_xyz, dt)
    pred_mean_speed = mean_speed_mps(pred_best_xyz, dt)
    speed_error = float(pred_mean_speed - gt_mean_speed)

    gt_yaw = global_motion_yaw_deg(gt_xyz)
    pred_yaw = global_motion_yaw_deg(pred_best_xyz)
    if np.isfinite(gt_yaw) and np.isfinite(pred_yaw):
        yaw_error = float(np.rad2deg(wrap_to_pi(np.deg2rad(pred_yaw - gt_yaw))))
    else:
        yaw_error = np.nan

    df_metrics = pd.DataFrame([{
        'best_sample_idx': best_idx,
        'minADE_m': min_ade,
        'final_point_error_m': final_point_error,
        'gt_final_x': float(gt_final_xy[0]),
        'gt_final_y': float(gt_final_xy[1]),
        'pred_final_x': float(pred_final_xy[0]),
        'pred_final_y': float(pred_final_xy[1]),
        'gt_mean_speed_mps': gt_mean_speed,
        'pred_mean_speed_mps': pred_mean_speed,
        'speed_error_mps': speed_error,
        'gt_yaw_deg': gt_yaw,
        'pred_yaw_deg': pred_yaw,
        'yaw_error_deg': yaw_error,
        'cot': str(cot_value),
    }])

    return df_metrics, best_idx, pred_best_xyz


def _compute_adaptive_xy_limits(*arrays, min_span=20.0, pad_ratio=0.12, pad_min=2.0):
    pts = [np.array([[0.0, 0.0]], dtype=np.float32)]
    for arr in arrays:
        if arr is None:
            continue
        pts.append(arr[:, :2])

    pts = np.concatenate(pts, axis=0)
    xmin, ymin = pts.min(axis=0)
    xmax, ymax = pts.max(axis=0)
    xspan = xmax - xmin
    yspan = ymax - ymin
    span = max(float(xspan), float(yspan), float(min_span))
    xcenter = 0.5 * (xmin + xmax)
    ycenter = 0.5 * (ymin + ymax)
    pad = max(span * pad_ratio, pad_min)
    half = 0.5 * span + pad
    return (xcenter - half, xcenter + half), (ycenter - half, ycenter + half)

### Load model and processor

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_enable_fp32_cpu_offload=False,
)

print('MODEL_PATH =', MODEL_PATH)
print('MODEL_PATH exists =', MODEL_PATH.exists())

model = Alpamayo1_5.from_pretrained(
    str(MODEL_PATH),
    dtype=torch.bfloat16,
    device_map='cuda:0',
    quantization_config=quantization_config,
)

processor = helper.get_processor(
    model.tokenizer,
    processor_name_or_path=PROCESSOR_PATH,
)

print('Model and processor loaded.')

### Load local offline data

In [ ]:
data = load_offline_dataset(
    clip_dir=CLIP_DIR,
    t0_sod=T0_SOD,
    num_history_steps=NUM_HISTORY_STEPS,
    num_future_steps=NUM_FUTURE_STEPS,
    time_step=TIME_STEP,
    num_frames=NUM_FRAMES,
    fps=FPS,
    frame0_gps_time_sod=FRAME0_GPS_TIME_SOD,
    debug=True,
    image_size=IMAGE_SIZE,
)

print('Offline dataset loaded.')
print('camera_indices:', data['camera_indices'].tolist())
print('image_frames shape:', tuple(data['image_frames'].shape))
print('ego_history_xyz shape:', tuple(data['ego_history_xyz'].shape))
print('ego_history_rot shape:', tuple(data['ego_history_rot'].shape))
print('ego_future_xyz shape:', tuple(data['ego_future_xyz'].shape))
print('ego_future_rot shape:', tuple(data['ego_future_rot'].shape))
print('requested frame indices:', data['requested_video_frame_indices'][0].tolist())
print('actual frame indices (first camera):', data['actual_video_frame_indices'][0].tolist())
print('num_clipped_frames_total:', data['num_clipped_frames_total'])
print('coordinate_frame_summary:', data['coordinate_frame_summary'])

### Time alignment diagnostics

In [ ]:
summary = data['time_alignment_summary']
df_time_alignment = pd.DataFrame([summary])

print('[Time alignment summary]')
display(df_time_alignment)

camera_indices = data['camera_indices'].cpu().tolist()
camera_names = [helper.CAMERA_DISPLAY_NAMES.get(i, f'Camera {i}') for i in camera_indices]

requested_ts = data['requested_image_timestamps_sod'].cpu().numpy()
requested_idx = data['requested_video_frame_indices'].cpu().numpy()
actual_idx = data['actual_video_frame_indices'].cpu().numpy()
clipped_mask = data['clipped_frame_mask'].cpu().numpy()
num_clipped = data['num_clipped_frames_per_camera'].cpu().numpy()
video_num_frames = data['video_num_frames'].cpu().numpy()

rows = []
for cam_row, cam_name in enumerate(camera_names):
    rows.append({
        'camera_index': camera_indices[cam_row],
        'camera_name': cam_name,
        'video_num_frames': int(video_num_frames[cam_row]),
        'requested_timestamps_sod': requested_ts[cam_row].tolist(),
        'requested_frame_indices': requested_idx[cam_row].tolist(),
        'actual_frame_indices': actual_idx[cam_row].tolist(),
        'clipped_frame_mask': clipped_mask[cam_row].tolist(),
        'num_clipped_frames': int(num_clipped[cam_row]),
    })

df_camera_time_diag = pd.DataFrame(rows)

print('[Per-camera frame alignment diagnostics]')
display(df_camera_time_diag)

### Diagnose history motion / yaw consistency

In [ ]:
hist_xyz = data['ego_history_xyz'].cpu().numpy()[0, 0, :, :]
hist_rot = data['ego_history_rot'].cpu().numpy()[0, 0, :, :, :]

df_hist_consistency = history_consistency_table(hist_xyz, hist_rot, TIME_STEP)

print('[History consistency summary]')
display(pd.DataFrame([{
    'mean_abs_yaw_minus_dxy_deg': float(df_hist_consistency['abs_yaw_minus_dxy_deg'].mean()),
    'median_abs_yaw_minus_dxy_deg': float(df_hist_consistency['abs_yaw_minus_dxy_deg'].median()),
    'last3_mean_abs_yaw_minus_dxy_deg': float(df_hist_consistency['abs_yaw_minus_dxy_deg'].tail(3).mean()),
    'last5_mean_abs_yaw_minus_dxy_deg': float(df_hist_consistency['abs_yaw_minus_dxy_deg'].tail(5).mean()),
}]))

print('\n[History consistency tail]')
display(df_hist_consistency.tail(5))

### Show all sampled images used for inference

In [ ]:
image_frames = data['image_frames'].cpu().numpy()
camera_indices = data['camera_indices'].cpu().tolist()
requested_indices = data['requested_video_frame_indices'].cpu().numpy()
actual_indices = data['actual_video_frame_indices'].cpu().numpy()
clipped_mask = data['clipped_frame_mask'].cpu().numpy()

num_cams, num_frames, _, _, _ = image_frames.shape
camera_names = [helper.CAMERA_DISPLAY_NAMES.get(i, f'Camera {i}') for i in camera_indices]

fig, axes = plt.subplots(num_cams, num_frames, figsize=(4 * num_frames, 3.8 * num_cams))
if num_cams == 1 and num_frames == 1:
    axes = np.array([[axes]])
elif num_cams == 1:
    axes = np.array([axes])
elif num_frames == 1:
    axes = np.array([[ax] for ax in axes])

for cam_idx in range(num_cams):
    for t_idx in range(num_frames):
        ax = axes[cam_idx, t_idx]
        frame = np.transpose(image_frames[cam_idx, t_idx], (1, 2, 0))
        ax.imshow(frame)
        ax.axis('off')
        clip_tag = 'CLIPPED' if clipped_mask[cam_idx, t_idx] else 'ok'
        ax.set_title(
            f'{camera_names[cam_idx]}\n'
            f't={t_idx}, req={requested_indices[cam_idx, t_idx]}, '
            f'actual={actual_indices[cam_idx, t_idx]} ({clip_tag})'
        )

plt.tight_layout()
plt.show()

### Prepare chat template

In [ ]:
messages = helper.create_message(
    frames=data['image_frames'].flatten(0, 1),
    camera_indices=data['camera_indices'],
)

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False,
    continue_final_message=True,
    return_dict=True,
    return_tensors='pt',
)

print('seq length:', inputs.input_ids.shape)

model_inputs = {
    'tokenized_data': inputs,
    'ego_history_xyz': data['ego_history_xyz'],
    'ego_history_rot': data['ego_history_rot'],
}
model_inputs = helper.to_device(model_inputs, DEVICE)

### Run inference

In [ ]:
if DEVICE.startswith('cuda'):
    torch.cuda.manual_seed_all(42)
    autocast_context = torch.autocast('cuda', dtype=torch.bfloat16)
else:
    autocast_context = torch.autocast(device_type=DEVICE, enabled=False)

with autocast_context:
    pred_xyz, pred_rot, extra = model.sample_trajectories_from_data_with_vlm_rollout(
        data=model_inputs,
        top_p=TOP_P,
        temperature=TEMPERATURE,
        num_traj_samples=NUM_TRAJ_SAMPLES,
        max_generation_length=MAX_GENERATION_LENGTH,
        return_extra=True,
    )

print('Inference done.')

### Show CoT and compute metrics in the unified model/action frame

In [ ]:
cot_value = extra['cot'][0]
if hasattr(cot_value, 'tolist'):
    cot_value = cot_value.tolist()

print('Chain-of-Causation:')
print(cot_value)

hist_xyz_plot = data['ego_history_xyz'].cpu().numpy()[0, 0, :, :]
gt_xyz_plot = data['ego_future_xyz'].cpu().numpy()[0, 0, :, :]
pred_xyz_np = pred_xyz.cpu().numpy()[0, 0, :, :, :]

df_metrics, best_idx, pred_best_xyz = summarize_main_metrics(
    gt_xyz=gt_xyz_plot,
    pred_xyz_np=pred_xyz_np,
    dt=TIME_STEP,
    cot_value=cot_value,
)

print('[Main metrics - unified model/action frame]')
display(df_metrics)

### Segment speed profile

In [ ]:
df_speed_profile = segment_mean_speed_table(
    gt_xyz=gt_xyz_plot,
    model_xyz=pred_best_xyz,
    dt=TIME_STEP,
    segment_sec=1.0,
)

print('[Segment speed profile - unified model/action frame]')
display(df_speed_profile)

### Plot final trajectory comparison

In [ ]:
xlim, ylim = _compute_adaptive_xy_limits(
    hist_xyz_plot,
    gt_xyz_plot,
    pred_best_xyz,
    min_span=20.0,
    pad_ratio=0.12,
    pad_min=2.0,
)

plt.figure(figsize=(7, 7))
plt.plot(hist_xyz_plot[:, 0], hist_xyz_plot[:, 1], 'b-o', linewidth=2, markersize=3, label='History')
plt.plot(gt_xyz_plot[:, 0], gt_xyz_plot[:, 1], 'k-o', linewidth=2.5, markersize=3.5, label='GT')
plt.plot(pred_best_xyz[:, 0], pred_best_xyz[:, 1], 'r-o', linewidth=2.5, markersize=3.5, label='Pred best')
plt.scatter([0.0], [0.0], c='red', marker='x', s=120, label='t0 ego')
plt.xlabel('x')
plt.ylabel('y')
plt.title(f'Offline inference @ t0_sod={T0_SOD}, minADE={float(df_metrics.iloc[0]["minADE_m"]):.3f}m')
plt.xlim(*xlim)
plt.ylim(*ylim)
plt.legend()
plt.grid(True, alpha=0.3)
plt.gca().set_aspect('equal', adjustable='box')
plt.tight_layout()
plt.show()